# Clonar TU voz a PiperEntrena una voz Piper con tus propias grabaciones. Usás el mismo guion que descargaste (el corpus de 1.300 frases): grabás las que quieras (300, 500 o las 1.300) y este notebook arma el dataset y entrena.**Cómo grabar (importante):** un clip corto por frase, nombrado con el número de la frase del corpus:- `f00001.wav` = tu voz leyendo la **línea 1** del corpus- `f00002.wav` = la línea 2, etc.Podés grabar solo las primeras N frases (ej. f00001 a f00300). El notebook usa solo las que subas. Comprimí la carpeta `wavs/` en **dataset.zip**.(El grabador guiado de LoudVox produce estos archivos automáticamente; pedíselo cuando quieras grabar.)**Requisitos de audio:** WAV mono, voz clara, mismo micrófono/distancia, lugar silencioso. Cada clip 1-15 s.**Cuánto grabar:** desde ~300 frases (~30-45 min de tu voz) sale una voz reconocible; 1.300 = máxima fidelidad.**Antes de correr:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.

In [ ]:
#@title 1. GPUimport torchassert torch.cuda.is_available(), "Activá T4 GPU: Entorno de ejecución -> Cambiar tipo de entorno"print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Descargar el guion (corpus) para las transcripciones!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"CORPUS = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]print(len(CORPUS), "frases en el guion. Grabá f00001.wav = frase 1, etc.")

In [ ]:
#@title 3. Subir tu dataset.zip (carpeta wavs/ con f00001.wav, f00002.wav...)from google.colab import filesimport zipfile, os, globup = files.upload()  # elegí tu dataset.zipname = list(up.keys())[0]os.makedirs("/content/dataset/wavs", exist_ok=True)with zipfile.ZipFile(name) as z: z.extractall("/content/unzip")# buscar los wav estén donde estén dentro del zipwavs = glob.glob("/content/unzip/**/*.wav", recursive=True)print(f"{len(wavs)} audios encontrados en el zip")

In [ ]:
#@title 4. Armar el dataset (empareja cada audio con su frase por el número)import re, wave, numpy as np, os, shutildef to_2205016mono(src, dst):    with wave.open(src, "rb") as w:        n, rate, ch, sw = w.getnframes(), w.getframerate(), w.getnchannels(), w.getsampwidth()        raw = w.readframes(n)    data = np.frombuffer(raw, dtype={1:np.int8,2:np.int16,4:np.int32}[sw]).astype(np.float32)    if sw != 2: data = data / (2**(8*sw-1)) * 32767    if ch > 1: data = data.reshape(-1, ch).mean(axis=1)    if rate != 22050:        idx = np.linspace(0, len(data)-1, int(len(data)*22050/rate))        data = np.interp(idx, np.arange(len(data)), data)    data = np.clip(data, -32768, 32767).astype(np.int16)    with wave.open(dst, "wb") as w:        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(data.tobytes())    return len(data)/22050rows, total, saltadas = [], 0.0, 0for src in sorted(wavs):    m = re.search(r"f?(\d{3,6})", os.path.basename(src))    if not m: saltadas += 1; continue    idx = int(m.group(1))    if not (1 <= idx <= len(CORPUS)): saltadas += 1; continue    frase = CORPUS[idx-1]    n = f"f{idx:05d}"    seg = to_2205016mono(src, f"/content/dataset/wavs/{n}.wav")    if not 1.0 <= seg <= 20.0: saltadas += 1; continue    rows.append(f"{n}|{frase}"); total += segopen("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")print(f"DATASET: {len(rows)} clips emparejados, {total/60:.1f} min de tu voz")if saltadas: print(f"({saltadas} archivos salteados: nombre sin número, fuera de rango o duración inválida)")assert rows, "Ningún audio pudo emparejarse. Revisá que se llamen f00001.wav, f00002.wav...

In [ ]:
#@title 5. Instalar Piper%cd /content!git clone -q https://github.com/rhasspy/piper.git%cd /content/piper/src/python!pip install -q -e .!pip install -q "pytorch-lightning~=1.9" espeak-phonemizer librosa "numpy<2"!apt-get install -yq espeak-ng > /dev/null!bash build_monotonic_align.shprint("Piper listo")

In [ ]:
#@title 6. Preprocesar + checkpoint base%cd /content/piper/src/python!python -m piper_train.preprocess --language es --input-dir /content/dataset \  --output-dir /content/train_out --dataset-format ljspeech --single-speaker --sample-rate 22050!wget -q -nc -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"print("Listo para entrenar")

In [ ]:
#@title 7. Entrenar (2-4 h; con menos audios, menos epochs alcanza)%cd /content/piper/src/python!python -m piper_train --dataset-dir /content/train_out --accelerator gpu --devices 1 \  --batch-size 16 --validation-split 0.0 --num-test-examples 0 \  --max_epochs 3219 --resume_from_checkpoint /content/base.ckpt \  --checkpoint-epochs 5 --precision 32 --quality medium

In [ ]:
#@title 8. Exportar y descargar tu vozimport glob, shutil, osck = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)assert ck, "Faltan checkpoints: corré la celda 7 unos epochs primero"%cd /content/piper/src/python!python -m piper_train.export_onnx "{ck[-1]}" /content/mi_voz.onnxshutil.copy("/content/train_out/config.json", "/content/mi_voz.onnx.json")from google.colab import filesfiles.download("/content/mi_voz.onnx")files.download("/content/mi_voz.onnx.json")print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'mi_voz' en Configuración")